# Cumulative unfolding animation — cross section & generator comparison

Wiener-SVD unfolding on **cumulative beam-time slices** (15 steps), following
`unfolding.ipynb` / `unfolding-data.ipynb` for the unfold pipeline and
`generator_comparison.ipynb` / `genie_flat_helpers.plot_generator_comparison`
for generator overlays.

For each cumulative exposure step:
- unfold beam-quality data vs GENIE CCQE MC
- save per-step `unfold_cache` to a `.pkl` file
- render unfolded differential cross section and generator-comparison frames
- compile GIFs (fixed- and auto-y-axis) with a beam-time progress bar

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pickle
import tempfile
from datetime import datetime
from functools import partial
from os import makedirs, path
from pathlib import Path

import imageio.v3 as iio
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import uproot
from PIL import Image

import sys
sys.path.append('/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana')
from pyanalib.split_df_helpers import load_dfs
from pyanalib.split_df_helpers_new import dfs_from_dir
from pyanalib.variable_calculator import get_cc1p0pi_tki
from pyanalib.pandas_helpers import pad_column_name

from analysis_village.numucc_1p0pi.variable_configs import VariableConfig
from analysis_village.numucc_1p0pi.utils import *
from analysis_village.numucc_1p0pi import utils as numucc_utils
from analysis_village.numucc_1p0pi.files_config import save_fig_base_dir
from analysis_village.numucc_1p0pi.constants import *
from analysis_village.numucc_1p0pi.dataset_locations import PLOTS_BASE, GENIE_GROUP_GLOBS
from analysis_village.numucc_1p0pi.syst_disk_layout import category_summary_npz_path
from analysis_village.numucc_1p0pi.genie_flat_helpers import (
    BRANCHES_NU,
    BRANCHES_TRK,
    GENIE_VAR_MAP,
    GIBUU_EXTRA_SCALE,
    add_genie_tki_columns,
    plot_generator_comparison,
    prepare_genie_trk_df,
)
from analysis_village.unfolding.wienersvd import *
from analysis_village.flux.raytrace_volume_defs import FV_SPLIT_TRUNCY_BOXES, RAYTRACE_VOLUME_LABEL
from pyanalib.covariance import *

import warnings
from pandas.errors import PerformanceWarning
warnings.filterwarnings("ignore", category=PerformanceWarning)
plt.style.use("presentation.mplstyle")
numucc_utils.fig_ext = ".png"

In [3]:
# --- paths & animation settings ---
DFS_ROOT = "/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs"
QUALITY_DF_PATH = path.join(DFS_ROOT, "2026_05_16_230705__sel_mup-data-1e20/merged_perTPC/beam_data_1e20_qualitycut.df")
DIR_GENIE_CCQE = path.dirname(GENIE_GROUP_GLOBS["CCQE"])

KEYS2LOAD_MC = ["hdr", "evt", "mcnu"]
N_MAX_CONCAT = 999

FOM_POT_SCALE = 0.9822
N_TIME_SPLITS = 15
GIF_FRAME_MS = 500
GIF_FPS = int(round(1000 / GIF_FRAME_MS))

C_type = 2
Norm_type = 0.0
breakdown_type = "topology"
UNFOLDING_SYST_KIND = "xsec"

FLUX_FILE = "/exp/sbnd/data/users/munjung/flux/SBND_gsimple_raytrace/Gen1.root"
GENIE_REF_POT = 1.0e20
GENIE_FLAT_DIR = "/pnfs/sbnd/persistent/users/apapadop/GENIETweakedSamples/v3_6_2_AR23_20i_00_000_gen1_flux"
GENIE_FLAT_FILES = {
    "GENIE AR23 (CC)": GENIE_FLAT_DIR + "/14_1000180400_CC_v3_6_2_AR23_20i_00_000.flat.root",
}

var_configs = [
    VariableConfig.tki_del_Tp(),
    # VariableConfig.tki_del_alpha(),
]

today_str = datetime.now().strftime("%Y%m%d")
work_dir = path.join(save_fig_base_dir, f"unfolding_fancyplot_{today_str}")
pkl_dir = path.join(work_dir, "pkl")
gif_out_dir = path.join(save_fig_base_dir, "gifs", f"unfolding_fancyplot_{today_str}")
for d in (work_dir, pkl_dir, gif_out_dir):
    makedirs(d, exist_ok=True)

print("quality-cut df:", QUALITY_DF_PATH)
print("work dir:", work_dir)
print("pkl dir:", pkl_dir)
print("gif dir:", gif_out_dir)

quality-cut df: /pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_16_230705__sel_mup-data-1e20/merged_perTPC/beam_data_1e20_qualitycut.df
work dir: /exp/sbnd/data/users/munjung/plots/numucc1p0pi/unfolding_fancyplot_20260611
pkl dir: /exp/sbnd/data/users/munjung/plots/numucc1p0pi/unfolding_fancyplot_20260611/pkl
gif dir: /exp/sbnd/data/users/munjung/plots/numucc1p0pi/gifs/unfolding_fancyplot_20260611


## Load GENIE CCQE MC and beam-quality data

In [4]:
genie_dfs = dfs_from_dir(
    DIR_GENIE_CCQE,
    filename_str="sel_mup-wgts_genie_CCQE",
    keys2load=KEYS2LOAD_MC,
    n_max_concat=N_MAX_CONCAT,
)
mc_evt_df_base = genie_dfs["evt"]
mc_hdr_df = genie_dfs["hdr"]
mc_nu_df_base = genie_dfs["mcnu"]
mc_tot_pot = mc_hdr_df["pot"].sum()

quality_dfs = load_dfs(QUALITY_DF_PATH, keys2load=["hdr", "trigger", "evt_good"], n_max_concat=1)
data_evt_df_base = quality_dfs["evt_good"]
data_hdr_df = quality_dfs["hdr"].join(quality_dfs["trigger"])
data_evt_df_base[("mc", "iscc")] = 999

print(f"evt_good rows: {len(data_evt_df_base):,}")
print(f"GENIE CCQE: evt={len(mc_evt_df_base):,}  mcnu={len(mc_nu_df_base):,}  mc_tot_pot={mc_tot_pot:.3e}")

Found 19 files to process
Files to process: ['/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_11_024530__sel_mup-wgts_genie_CCQE/merged_perTPC/2026_05_11_024530__sel_mup-wgts_genie_CCQE_merged_0000.df', '/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_11_024530__sel_mup-wgts_genie_CCQE/merged_perTPC/2026_05_11_024530__sel_mup-wgts_genie_CCQE_merged_0002.df', '/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_11_024530__sel_mup-wgts_genie_CCQE/merged_perTPC/2026_05_11_024530__sel_mup-wgts_genie_CCQE_merged_0003.df', '/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_11_024530__sel_mup-wgts_genie_CCQE/merged_perTPC/2026_05_11_024530__sel_mup-wgts_genie_CCQE_merged_0004.df', '/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_11_024530__sel_mup-wgts_genie_CCQE/merged_perTPC/2026_05_11_024530__sel_mup-wgts_genie_CCQE_merged_0005.df', '/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_11_024530__sel_mup-wgts_genie_CCQE/merged_perTPC/2026_05

100%|██████████| 19/19 [01:53<00:00,  5.95s/it]


REMEMBER TO RECALCULATE TKI AND CHECK FV!!
evt_good rows: 12,804
GENIE CCQE: evt=70,535  mcnu=10,933,450  mc_tot_pot=5.347e+20


In [5]:
perTPC_inset = 10

def perTPC_cut(df):
    in_TPC1 = (
        InFV(df.slc.vertex, det="SBND_TPC1", incathode=perTPC_inset)
        & InFV(df.mu.pfp.trk.end, det="SBND_TPC1", incathode=perTPC_inset)
        & InFV(df.p.pfp.trk.end, det="SBND_TPC1", incathode=perTPC_inset)
    )
    in_TPC2 = (
        InFV(df.slc.vertex, det="SBND_TPC2", incathode=perTPC_inset)
        & InFV(df.mu.pfp.trk.end, det="SBND_TPC2", incathode=perTPC_inset)
        & InFV(df.p.pfp.trk.end, det="SBND_TPC2", incathode=perTPC_inset)
    )
    return in_TPC1 | in_TPC2


def evt_df_fixed(df):
    slc_mudf = df.mu.pfp.trk
    slc_pdf = df.p.pfp.trk
    tki_reco = get_cc1p0pi_tki(
        slc_mudf,
        slc_pdf,
        pad_column_name(("P", "p_muon"), slc_mudf),
        pad_column_name(("P", "p_proton"), slc_pdf),
    )
    df["del_Tp_x"] = tki_reco["del_Tp_x"]
    df["del_Tp_y"] = tki_reco["del_Tp_y"]

    mc_mudf = df.mu.pfp.trk.truth.p
    mc_pdf = df.p.pfp.trk.truth.p
    tki_mc = get_cc1p0pi_tki(
        mc_mudf,
        mc_pdf,
        pad_column_name(("totp",), mc_mudf),
        pad_column_name(("totp",), mc_pdf),
    )
    df["mc_del_Tp_x"] = tki_mc["del_Tp_x"]
    df["mc_del_Tp_y"] = tki_mc["del_Tp_y"]
    df[("mc", "del_Tp_x")] = tki_mc["del_Tp_x"]
    df[("mc", "del_Tp_y")] = tki_mc["del_Tp_y"]

    n_before = len(df)
    df = df[np.abs(df.slc.vertex.x) > 10]
    if "topo_categ" not in df.columns:
        df = df.copy()
        df.loc[:, "topo_categ"] = get_topo_category(df)
    return df, n_before


data_evt_df_base = data_evt_df_base.loc[perTPC_cut(data_evt_df_base)]
mc_evt_df_base = mc_evt_df_base.loc[perTPC_cut(mc_evt_df_base)]
mc_evt_df_base, _ = evt_df_fixed(mc_evt_df_base)
data_evt_df_base, _ = evt_df_fixed(data_evt_df_base)

if "topo_categ" not in mc_nu_df_base.columns:
    mc_nu_df_base = mc_nu_df_base.copy()
    mc_nu_df_base.loc[:, "topo_categ"] = get_topo_category(mc_nu_df_base)

print(f"perTPC selected: data={len(data_evt_df_base):,} mc={len(mc_evt_df_base):,}")

perTPC selected: data=12,804 mc=70,535


/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana/envs/venv_py310_cafpyana/lib/python3.10/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in arccos
  result = getattr(ufunc, method)(*inputs, **kwargs)


## Flux, systematics, and GENIE flat ROOT

In [6]:
print(f"Fiducial volume (FV_split_truncY): {RAYTRACE_VOLUME_LABEL['FV_split_truncY']}")
V_SBND = 0.0
for box in FV_SPLIT_TRUNCY_BOXES:
    V_SBND += (box["x_range"][1] - box["x_range"][0]) * (box["y_range"][1] - box["y_range"][0]) * (box["z_range"][1] - box["z_range"][0])
print(f"V_SBND = {V_SBND:.6e} cm³")

integrated_flux_per_pot = get_integrated_flux(FLUX_FILE, plot=False)
N_TARGETS = (RHO * V_SBND / M_AR) * N_A
print(f"integrated flux per POT = {integrated_flux_per_pot:.6e} ν/cm²")
print(f"N_targets = {N_TARGETS:.4e}")


def xsec_unit_for_pot(data_tot_pot):
    integrated_flux = integrated_flux_per_pot * data_tot_pot
    return 1.0 / (integrated_flux * N_TARGETS)


SYST_DISK_ROOT = "/exp/sbnd/data/users/munjung/plots/numucc1p0pi/systematics-final"
CATEGORY_SUMMARY_OUT = Path(PLOTS_BASE) / "syst_uncertainty_breakdown" / "final_selected" / "category_syst_summary.npz"
CATEGORY_SUMMARY_NPZ = category_summary_npz_path(SYST_DISK_ROOT)
if not path.isfile(CATEGORY_SUMMARY_NPZ) and CATEGORY_SUMMARY_OUT.is_file():
    CATEGORY_SUMMARY_NPZ = str(CATEGORY_SUMMARY_OUT)

import os
os.environ["NUMUCC_SYST_DISK_ROOT"] = SYST_DISK_ROOT
print("category summary:", CATEGORY_SUMMARY_NPZ, "exists =", path.isfile(CATEGORY_SUMMARY_NPZ))


def get_syst_cov(var_config):
    return get_category_summary_syst_unc(
        var_config,
        syst_kind=UNFOLDING_SYST_KIND,
        syst_disk_root=SYST_DISK_ROOT,
        category_syst_summary_path=CATEGORY_SUMMARY_NPZ,
    )[1]

Fiducial volume (FV_split_truncY): $10<|x|<190$: $|y|<190$ ($10<z<250$), $-190<y<100$ ($250<z<450$)
V_SBND = 5.371200e+07 cm³
Integrated flux: 1.518e-08
integrated flux per POT = 1.518012e-08 ν/cm²
N_targets = 1.1203e+30
category summary: /exp/sbnd/data/users/munjung/plots/numucc1p0pi/systematics-final/CategorySummary/category_syst_summary.npz exists = True


In [7]:
# generator_comparison.ipynb: flat-file TKI branches are dpt, dalphat, dphit
# (dpt in MeV -> divide by 1e3; angles in rad -> multiply by 180/pi).
GENIE_VAR_MAP.update({
    "tki-del_Tp": ("dpt", 1e-3),
    "tki-del_alpha": ("dalphat", 180.0 / np.pi),
    "tki-del_phi": ("dphit", 180.0 / np.pi),
    "muon-dir_z": ("mu_dir_z", 1.0),
    "proton-dir_z": ("proton_dir_z", 1.0),
})


def _genie_add_trk_counts(nudf, trkdf):
    nudf = nudf.copy()
    trkdf = trkdf.copy()
    trkdf["_p"] = np.sqrt(trkdf.px**2 + trkdf.py**2 + trkdf.pz**2)
    for pid, pname, pth in zip([13, 2212, 211, 111], ["mu", "proton", "pi", "pi0"], [0.22, 0.3, 0.07, 0]):
        ntrks = trkdf[(np.abs(trkdf.pdg) == pid) & (trkdf._p > pth)].pdg.groupby(level=[0]).count()
        nudf[f"n{pname}s"] = ntrks.fillna(0)
        species = trkdf[trkdf.pdg == pid] if pname == "proton" else trkdf[np.abs(trkdf.pdg) == pid]
        leading = species.sort_values("_p", ascending=False).groupby(level=[0]).head(1)
        leading_p = leading["_p"]
        leading_p.name = f"{pname}_p"
        nudf = nudf.join(leading_p.reset_index(level=[1])[f"{pname}_p"])
        leading_p_dir_z = leading.pz / leading_p
        leading_p_dir_z.name = f"{pname}_dir_z"
        nudf = nudf.join(leading_p_dir_z.reset_index(level=[1])[f"{pname}_dir_z"])
    return nudf


def _genie_flat_signal_mask(nudf):
    numu_cc = (nudf.PDGnu == 14) & (nudf.cc == 1)
    one_mu = numu_cc & (nudf.nmus == 1) & (nudf.mu_p < 1.0)
    one_p = one_mu & (nudf.nprotons == 1) & (nudf.proton_p < 1.0)
    return one_p & (np.nan_to_num(nudf.npis, nan=0) == 0) & (np.nan_to_num(nudf.npi0s, nan=0) == 0)


def load_genie_flat_pack(label, flat_path):
    print(f"Loading {label}: {flat_path}")
    events = uproot.open(flat_path + ":FlatTree_VARS")
    nu_df = events.arrays(BRANCHES_NU, library="pd")
    trk_df = prepare_genie_trk_df(events.arrays(BRANCHES_TRK, library="ak"))
    nu_df = _genie_add_trk_counts(nu_df, trk_df)
    nu_df = add_genie_tki_columns(nu_df)
    pot_scale = 1.05
    if "GiBUU" in flat_path or "gibuu" in flat_path.lower():
        pot_scale = pot_scale / GIBUU_EXTRA_SCALE
    sig_mask = _genie_flat_signal_mask(nu_df)
    print(f"  entries={len(nu_df):,}  1p0pi signal={int(sig_mask.sum()):,}")
    return {"nu_df": nu_df, "sig_mask": sig_mask, "pot_scale": pot_scale, "flat_path": flat_path}


genie_flat_cache_base = {
    label: load_genie_flat_pack(label, flat_path) for label, flat_path in GENIE_FLAT_FILES.items()
}

Loading GENIE AR23 (CC): /pnfs/sbnd/persistent/users/apapadop/GENIETweakedSamples/v3_6_2_AR23_20i_00_000_gen1_flux/14_1000180400_CC_v3_6_2_AR23_20i_00_000.flat.root
  entries=1,000,000  1p0pi signal=318,037


## Time-ordered cumulative slices

In [8]:
def split_hdr_time_chunks(hdr_df, n_splits=N_TIME_SPLITS):
    sorted_hdr = hdr_df.sort_values(["run", "evt"], kind="mergesort")
    return [sorted_hdr.iloc[idx] for idx in np.array_split(np.arange(len(sorted_hdr)), n_splits)]


def select_evt_for_hdr(evt_df, hdr_df):
    evt_base = evt_df.reset_index(level=[2])
    common = evt_base.index.intersection(hdr_df.index)
    return evt_base.loc[common].reset_index().set_index(["__ntuple", "entry", "rec.slc..index"])


def cumulative_step_slice(step_idx, hdr_splits, evt_df):
    hdr_cum = pd.concat(hdr_splits[: step_idx + 1])
    evt_cum = select_evt_for_hdr(evt_df, hdr_cum)
    return hdr_cum, evt_cum


def format_time_ns(t_ns):
    return pd.Timestamp(t_ns, unit="ns", tz="UTC").strftime("%Y-%m-%d")


def build_time_progress_context(hdr_df, hdr_splits):
    sorted_hdr = hdr_df.sort_values(["run", "evt"], kind="mergesort")
    t_min_ns = sorted_hdr["global_trigger_time"].min()
    t_max_ns = sorted_hdr["global_trigger_time"].max()
    span_ns = t_max_ns - t_min_ns
    chunk_edge_fracs = [
        (hdr_chunk["global_trigger_time"].max() - t_min_ns) / span_ns for hdr_chunk in hdr_splits
    ]
    return {"t_min_ns": t_min_ns, "t_max_ns": t_max_ns, "span_ns": span_ns, "chunk_edge_fracs": chunk_edge_fracs}


def time_progress_for_hdr_cum(hdr_cum, time_ctx):
    t_current_ns = hdr_cum["global_trigger_time"].max()
    progress_frac = (t_current_ns - time_ctx["t_min_ns"]) / time_ctx["span_ns"]
    return float(np.clip(progress_frac, 0.0, 1.0)), int(t_current_ns)


def prepare_step_frames(hdr_cum):
    data_tot_pot = hdr_cum["pot"].sum() * FOM_POT_SCALE
    mc_pot_scale = data_tot_pot / mc_tot_pot
    data_evt_df = select_evt_for_hdr(data_evt_df_base, hdr_cum).copy()
    data_evt_df["pot_weight"] = np.ones(len(data_evt_df))
    mc_evt_df = mc_evt_df_base.copy()
    mc_nu_df = mc_nu_df_base.copy()
    mc_evt_df["pot_weight"] = mc_pot_scale * np.ones(len(mc_evt_df))
    mc_nu_df["pot_weight"] = mc_pot_scale * np.ones(len(mc_nu_df))
    xsec_unit = xsec_unit_for_pot(data_tot_pot)
    return data_evt_df, mc_evt_df, mc_nu_df, data_tot_pot, xsec_unit, mc_pot_scale


def genie_flat_cache_for_pot(data_tot_pot, ref_pot):
    pot_ratio = data_tot_pot / ref_pot
    scaled = {}
    for label, pack in genie_flat_cache_base.items():
        p = dict(pack)
        p["pot_scale"] = pack["pot_scale"] * pot_ratio
        scaled[label] = p
    return scaled


hdr_splits = split_hdr_time_chunks(data_hdr_df, N_TIME_SPLITS)
time_ctx = build_time_progress_context(data_hdr_df, hdr_splits)
_, _, _, full_data_tot_pot, _, _ = prepare_step_frames(pd.concat(hdr_splits))
print(f"full cumulative POT = {full_data_tot_pot:.3e}")

full cumulative POT = 8.808e+19


## Unfolding, plotting, and GIF helpers

In [9]:
def pack_unfold_results(unfold, var_config):
    bins = var_config.bins
    bin_widths = np.diff(bins)
    if len(bins) == 2:
        bin_widths = np.array([1.0])
    unfolded = np.asarray(unfold["unfold"], dtype=float)
    return {
        "bins": np.asarray(bins, dtype=float),
        "bin_centers": np.asarray(var_config.bin_centers, dtype=float),
        "bin_widths": bin_widths,
        "unfold": unfolded,
        "unfold_per_bin_width": unfolded / bin_widths,
        "stat_err_per_bin_width": np.sqrt(np.maximum(np.diag(unfold["StatUnfoldCov"]), 0.0)) / bin_widths,
        "total_err_per_bin_width": np.sqrt(np.maximum(np.diag(unfold["UnfoldCov"]), 0.0)) / bin_widths,
        "AddSmear": np.asarray(unfold["AddSmear"], dtype=float),
    }


def run_unfold_for_var(var_config, data_evt_df, mc_evt_df, mc_nu_df, xsec_unit, syst_cov_matrix):
    unfolding_plotter = partial(
        overlay_hists,
        breakdown_type=breakdown_type,
        mc_df=mc_evt_df,
        data_df=data_evt_df,
        plot=False,
        save_fig=False,
        syst_kind=UNFOLDING_SYST_KIND,
        syst_disk_root=SYST_DISK_ROOT,
        category_syst_summary_path=CATEGORY_SUMMARY_NPZ,
        load_syst_from_summary=False,
    )
    ret = unfolding_plotter(var_config=var_config, syst=syst_cov_matrix)
    ret_signal_hists = signal_hists(
        mc_evt_df, mc_nu_df, var_config, mode="unfold", return_data=True, plot=False, signal_truth_fv="none",
    )
    if len(var_config.bins) == 2:
        reco_vs_true = np.array([[1.0]])
    else:
        reco_vs_true, _, _ = np.histogram2d(
            ret_signal_hists["var_sel_truth"],
            ret_signal_hists["var_sel_reco"],
            weights=ret_signal_hists["wgt_sel_truth"],
            bins=var_config.bins,
        )
    eff = ret_signal_hists["nevts_sel_truth"] / ret_signal_hists["nevts_allmc"]
    response = get_response_matrix(reco_vs_true, eff)
    nevts_sel_data = ret["total_data"] - ret["total_mc_bkgd"]
    measured = nevts_sel_data * xsec_unit
    model = ret_signal_hists["nevts_allmc"] * xsec_unit
    covariance = cov_from_fraccov(syst_cov_matrix, ret_signal_hists["nevts_sel_reco"]) * xsec_unit**2
    unfold = WienerSVD(response, model, measured, covariance, C_type, Norm_type, stat_scaling=xsec_unit)
    return {
        "unfold": unfold,
        "model": model,
        "measured": measured,
        "ret_signal_hists": ret_signal_hists,
        "var_config": var_config,
        "pack": pack_unfold_results(unfold, var_config),
    }


def add_time_progress_bar(fig, progress_frac, time_ctx, t_current_ns):
    fig.subplots_adjust(bottom=0.20, top=0.90)
    ax = fig.add_axes([0.10, 0.012, 0.84, 0.075])
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    bar_y, bar_h = 0.42, 0.28
    ax.add_patch(mpatches.Rectangle((0, bar_y), 1, bar_h, facecolor="#ececec", edgecolor="0.45", linewidth=0.9, clip_on=False))
    ax.add_patch(mpatches.Rectangle((0, bar_y), progress_frac, bar_h, facecolor="steelblue", edgecolor="none", alpha=0.9, clip_on=False))
    for edge_frac in time_ctx["chunk_edge_fracs"][:-1]:
        ax.axvline(edge_frac, ymin=bar_y - 0.08, ymax=bar_y + bar_h + 0.08, color="0.55", linewidth=0.7, linestyle=":", clip_on=False)
    ax.axvline(progress_frac, ymin=bar_y - 0.18, ymax=bar_y + bar_h + 0.18, color="darkblue", linewidth=1.6, clip_on=False)
    ax.text(0.5, 0.92, "Beam time", ha="center", va="bottom", fontsize=10, fontweight="bold")
    ax.text(0, 0.92, format_time_ns(time_ctx["t_min_ns"]), ha="left", va="bottom", fontsize=9)
    ax.text(1, 0.92, format_time_ns(time_ctx["t_max_ns"]), ha="right", va="bottom", fontsize=9)
    label_x = min(max(progress_frac, 0.06), 0.94)
    ax.text(label_x, 0.08, format_time_ns(t_current_ns), ha="center", va="top", fontsize=9, color="darkblue")
    ax.text(progress_frac, bar_y + bar_h + 0.10, f"{100 * progress_frac:.1f}%", ha="center", va="bottom", fontsize=8.5, color="darkblue")


def _keep_figure_open(func, *args, **kwargs):
    _orig_close, _orig_show = plt.close, plt.show
    plt.close = lambda _fig=None: None
    plt.show = lambda *a, **k: None
    try:
        func(*args, **kwargs)
        return plt.gcf()
    finally:
        plt.close = _orig_close
        plt.show = _orig_show


def unfolded_ymax(cache, var_config):
    unfold = cache["unfold"]
    bin_widths = np.diff(var_config.bins)
    if len(var_config.bins) == 2:
        bin_widths = np.array([1.0])
    unfolded_pw = np.asarray(unfold["unfold"], dtype=float) / bin_widths
    model_pw = (unfold["AddSmear"] @ cache["model"]) / bin_widths
    return max(np.nanmax(unfolded_pw), np.nanmax(model_pw)) * 1.25


def render_unfolded_frame(cache, var_config, xsec_unit, save_path, ymax=None, time_ctx=None, progress_frac=None, t_current_ns=None):
    models = {"GENIE": [cache["model"], "C0"]}
    fig = _keep_figure_open(
        plot_unfolded_result,
        cache["unfold"], cache["measured"], models, var_config,
        xsec_unit=xsec_unit, plot=False, save_fig=False, data=True, approval="preliminary",
        textloc=[0.05, 0.55],
    )
    if ymax is not None:
        fig.axes[0].set_ylim(0.0, ymax)
    if time_ctx is not None:
        add_time_progress_bar(fig, progress_frac, time_ctx, t_current_ns)
    fig.savefig(save_path, bbox_inches="tight", dpi=numucc_utils.dpi)
    plt.close(fig)


def generator_ymax(cache, var_config, genie_flat_cache):
    unfold = cache["unfold"]
    bin_widths = np.diff(var_config.bins)
    if len(var_config.bins) == 2:
        bin_widths = np.array([1.0])
    unfolded_pw = np.asarray(unfold["unfold"], dtype=float) / bin_widths
    vals = [unfolded_pw]
    model_pw = (unfold["AddSmear"] @ cache["model"]) / bin_widths
    vals.append(model_pw)
    from analysis_village.numucc_1p0pi.genie_flat_helpers import genie_flat_differential_xsec
    for gpack in genie_flat_cache.values():
        sigma_bin, sigma_pw = genie_flat_differential_xsec(gpack["nu_df"], gpack["sig_mask"], var_config, gpack["pot_scale"])
        smeared = unfold["AddSmear"] @ sigma_bin
        vals.append(smeared / bin_widths)
    return float(np.nanmax(np.concatenate([np.ravel(v) for v in vals]))) * 1.7


def render_generator_frame(cache, var_config, xsec_unit, genie_flat_cache, save_path, ymax=None, time_ctx=None, progress_frac=None, t_current_ns=None):
    fig = _keep_figure_open(
        plot_generator_comparison,
        var_config, cache, genie_flat_cache, xsec_unit,
        save_fig=False, show_ratio=False, show_title=False, show_chi2=False, approval="preliminary",
    )
    if ymax is not None:
        fig.axes[0].set_ylim(0.0, ymax)
    if time_ctx is not None:
        add_time_progress_bar(fig, progress_frac, time_ctx, t_current_ns)
    fig.savefig(save_path, bbox_inches="tight", dpi=numucc_utils.dpi)
    plt.close(fig)


def make_gif(image_paths, output_path, fps=GIF_FPS):
    frames = [Image.fromarray(iio.imread(p)) for p in image_paths]
    frames[0].save(output_path, format="GIF", save_all=True, append_images=frames[1:], duration=1000 // fps, loop=0)
    print(f"GIF saved to {output_path}")

## Run cumulative unfolding, save PKLs, and build GIFs

In [ ]:
syst_cov_by_var = {vc.var_save_name: get_syst_cov(vc) for vc in var_configs}

for var_config in var_configs:
    print(f"\n=== {var_config.var_save_name} ===")

    # Pre-compute fixed y-axis limits from the final cumulative step
    hdr_final, _ = cumulative_step_slice(N_TIME_SPLITS - 1, hdr_splits, data_evt_df_base)
    data_final, mc_final, nu_final, _, xsec_final, _ = prepare_step_frames(hdr_final)
    cache_final = run_unfold_for_var(var_config, data_final, mc_final, nu_final, xsec_final, syst_cov_by_var[var_config.var_save_name])
    genie_flat_final = genie_flat_cache_for_pot(full_data_tot_pot, full_data_tot_pot)
    fixed_unfold_ymax = unfolded_ymax(cache_final, var_config)
    fixed_gen_ymax = generator_ymax(cache_final, var_config, genie_flat_final)

    with tempfile.TemporaryDirectory(prefix="unfold_fancyplot_") as tmp_root:
        frame_dirs = {
            "unfold_fixed": path.join(tmp_root, "unfold_fixed"),
            "unfold_auto": path.join(tmp_root, "unfold_auto"),
            "gen_fixed": path.join(tmp_root, "gen_fixed"),
            "gen_auto": path.join(tmp_root, "gen_auto"),
        }
        for d in frame_dirs.values():
            makedirs(d)
        frame_paths = {k: [] for k in frame_dirs}

        for step_idx in range(N_TIME_SPLITS):
            step_num = step_idx + 1
            hdr_cum, _ = cumulative_step_slice(step_idx, hdr_splits, data_evt_df_base)
            data_evt_df, mc_evt_df, mc_nu_df, data_tot_pot, xsec_unit, mc_pot_scale = prepare_step_frames(hdr_cum)
            progress_frac, t_current_ns = time_progress_for_hdr_cum(hdr_cum, time_ctx)
            genie_flat_cache = genie_flat_cache_for_pot(data_tot_pot, full_data_tot_pot)

            unfold_cache = {}
            cache = run_unfold_for_var(
                var_config, data_evt_df, mc_evt_df, mc_nu_df, xsec_unit, syst_cov_by_var[var_config.var_save_name],
            )
            unfold_cache[var_config.var_save_name] = cache

            step_meta = {
                "schema": "numucc1p0pi_unfold_cumulative_step_v1",
                "step_idx": step_idx,
                "step_num": step_num,
                "n_time_splits": N_TIME_SPLITS,
                "data_tot_pot": float(data_tot_pot),
                "xsec_unit": float(xsec_unit),
                "mc_pot_scale": float(mc_pot_scale),
                "progress_frac": float(progress_frac),
                "t_current_ns": int(t_current_ns),
                "n_data_evt": int(len(data_evt_df)),
            }
            pkl_path = path.join(pkl_dir, f"step_{step_idx:02d}_{var_config.var_save_name}_unfold_cache.pkl")
            with open(pkl_path, "wb") as f:
                pickle.dump({"meta": step_meta, "unfold_cache": unfold_cache}, f)

            time_kwargs = dict(time_ctx=time_ctx, progress_frac=progress_frac, t_current_ns=t_current_ns)
            tag = f"frame_{step_idx:02d}"

            unfold_auto = path.join(frame_dirs["unfold_auto"], f"{tag}.png")
            render_unfolded_frame(cache, var_config, xsec_unit, unfold_auto, ymax=None, **time_kwargs)
            frame_paths["unfold_auto"].append(unfold_auto)

            unfold_fixed = path.join(frame_dirs["unfold_fixed"], f"{tag}.png")
            render_unfolded_frame(cache, var_config, xsec_unit, unfold_fixed, ymax=fixed_unfold_ymax, **time_kwargs)
            frame_paths["unfold_fixed"].append(unfold_fixed)

            gen_auto = path.join(frame_dirs["gen_auto"], f"{tag}.png")
            render_generator_frame(cache, var_config, xsec_unit, genie_flat_cache, gen_auto, ymax=None, **time_kwargs)
            frame_paths["gen_auto"].append(gen_auto)

            gen_fixed = path.join(frame_dirs["gen_fixed"], f"{tag}.png")
            render_generator_frame(cache, var_config, xsec_unit, genie_flat_cache, gen_fixed, ymax=fixed_gen_ymax, **time_kwargs)
            frame_paths["gen_fixed"].append(gen_fixed)

            print(
                f"  step {step_num:2d}: N_data={len(data_evt_df):4d}  POT={get_pot_str(data_tot_pot)}  "
                f"time={100 * progress_frac:.1f}%  pkl={pkl_path}"
            )

        gif_map = {
            "unfold_fixed": f"{var_config.var_save_name}_unfolded_xsec_cumulative_fixed_axes.gif",
            "unfold_auto": f"{var_config.var_save_name}_unfolded_xsec_cumulative_auto_axes.gif",
            "gen_fixed": f"{var_config.var_save_name}_generator_comparison_cumulative_fixed_axes.gif",
            "gen_auto": f"{var_config.var_save_name}_generator_comparison_cumulative_auto_axes.gif",
        }
        for key, gif_name in gif_map.items():
            make_gif(frame_paths[key], path.join(gif_out_dir, gif_name), fps=GIF_FPS)


=== tki-del_Tp ===
No intime cosmics provided
No intime cosmics provided
  step  1: N_data= 849  POT=5.93$\times 10^{18}$  time=6.4%  pkl=/exp/sbnd/data/users/munjung/plots/numucc1p0pi/unfolding_fancyplot_20260611/pkl/step_00_tki-del_Tp_unfold_cache.pkl
No intime cosmics provided
  step  2: N_data=1720  POT=1.19$\times 10^{19}$  time=12.8%  pkl=/exp/sbnd/data/users/munjung/plots/numucc1p0pi/unfolding_fancyplot_20260611/pkl/step_01_tki-del_Tp_unfold_cache.pkl
No intime cosmics provided
  step  3: N_data=2599  POT=1.78$\times 10^{19}$  time=18.5%  pkl=/exp/sbnd/data/users/munjung/plots/numucc1p0pi/unfolding_fancyplot_20260611/pkl/step_02_tki-del_Tp_unfold_cache.pkl
No intime cosmics provided
  step  4: N_data=3466  POT=2.38$\times 10^{19}$  time=24.4%  pkl=/exp/sbnd/data/users/munjung/plots/numucc1p0pi/unfolding_fancyplot_20260611/pkl/step_03_tki-del_Tp_unfold_cache.pkl
No intime cosmics provided
  step  5: N_data=4310  POT=2.97$\times 10^{19}$  time=30.1%  pkl=/exp/sbnd/data/users/munj

: 